In [0]:
# Configuration

CATALOG = "worldbank_ai"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

SOURCE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.countries_raw"
)

TARGET_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.countries"
)

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# Imports

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# Load Bronze country data

bronze_countries_df = spark.table(
    SOURCE_TABLE
)

bronze_count = bronze_countries_df.count()

print(
    f"Bronze country records: "
    f"{bronze_count:,}"
)

bronze_countries_df.printSchema()

display(
    bronze_countries_df.limit(10)
)

In [0]:
# Inspect World Bank region metadata

display(
    bronze_countries_df
    .groupBy(
        "region_id",
        "region_name"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

In [0]:
# Validate source identifiers

null_entity_ids = (
    bronze_countries_df
    .filter(
        F.col("country_id").isNull()
    )
    .count()
)

duplicate_entity_ids = (
    bronze_countries_df
    .groupBy("country_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_entity_count = (
    duplicate_entity_ids.count()
)

print(
    f"Null country IDs: "
    f"{null_entity_ids}"
)

print(
    f"Duplicate country IDs: "
    f"{duplicate_entity_count}"
)

if null_entity_ids > 0:
    raise RuntimeError(
        "Null country IDs found in Bronze."
    )

if duplicate_entity_count > 0:

    display(
        duplicate_entity_ids.limit(20)
    )

    raise RuntimeError(
        "Duplicate country IDs found in Bronze."
    )

print(
    "Bronze country key validation passed."
)

In [0]:
# Inspect available Bronze columns

print("Bronze columns:")
print("-" * 60)

for column_name in bronze_countries_df.columns:
    print(column_name)

In [0]:
# Transform Bronze countries into Silver

silver_countries_df = (
    bronze_countries_df

    # --------------------------------------------------
    # Standardize identifiers and names
    # --------------------------------------------------

    .withColumn(
        "entity_id",
        F.trim(
            F.col("country_id")
        )
    )

    .withColumn(
        "iso2_code",
        F.when(
            F.trim(F.col("iso2_code")) == "",
            None
        ).otherwise(
            F.upper(
                F.trim(
                    F.col("iso2_code")
                )
            )
        )
    )

    .withColumn(
        "entity_name",
        F.trim(
            F.col("country_name")
        )
    )

    # --------------------------------------------------
    # Classify World Bank entity
    # --------------------------------------------------

    .withColumn(
        "entity_type",
        F.when(
            (
                F.col("region_id") == "NA"
            )
            & (
                F.col("region_name")
                == "Aggregates"
            ),
            F.lit("AGGREGATE")
        ).otherwise(
            F.lit("COUNTRY_OR_ECONOMY")
        )
    )

    # --------------------------------------------------
    # Clean descriptive metadata
    # --------------------------------------------------

    .withColumn(
        "region_name",
        F.when(
            F.trim(
                F.col("region_name")
            ) == "",
            None
        ).otherwise(
            F.trim(
                F.col("region_name")
            )
        )
    )

    .withColumn(
        "income_level_name",
        F.when(
            F.trim(
                F.col("income_level_name")
            ) == "",
            None
        ).otherwise(
            F.trim(
                F.col("income_level_name")
            )
        )
    )

    .withColumn(
        "capital_city",
        F.when(
            F.trim(
                F.col("capital_city")
            ) == "",
            None
        ).otherwise(
            F.trim(
                F.col("capital_city")
            )
        )
    )

    # --------------------------------------------------
    # Convert coordinates from Bronze strings
    # to numeric Silver fields
    # --------------------------------------------------

    .withColumn(
        "longitude",
        F.when(
            F.trim(
                F.col("longitude")
            ) == "",
            None
        ).otherwise(
            F.col("longitude")
            .cast(T.DoubleType())
        )
    )

    .withColumn(
        "latitude",
        F.when(
            F.trim(
                F.col("latitude")
            ) == "",
            None
        ).otherwise(
            F.col("latitude")
            .cast(T.DoubleType())
        )
    )

    # --------------------------------------------------
    # Silver processing metadata
    # --------------------------------------------------

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )

    # --------------------------------------------------
    # Select governed Silver schema
    # --------------------------------------------------

    .select(
        "entity_id",
        "iso2_code",
        "entity_name",
        "entity_type",

        "region_id",
        "region_name",

        "admin_region_id",
        "admin_region_name",

        "income_level_id",
        "income_level_name",

        "lending_type_id",
        "lending_type_name",

        "capital_city",

        "longitude",
        "latitude",

        "source_system",
        "source_endpoint",
        "ingested_at",

        "silver_processed_at"
    )
)

In [0]:
# Inspect Silver schema

silver_countries_df.printSchema()

display(
    silver_countries_df.limit(20)
)

In [0]:
# Validate entity classification

entity_type_summary_df = (
    silver_countries_df
    .groupBy(
        "entity_type"
    )
    .count()
    .orderBy(
        "entity_type"
    )
)

display(
    entity_type_summary_df
)

classified_count = (
    silver_countries_df.count()
)

if classified_count != bronze_count:

    raise RuntimeError(
        "Entity classification changed "
        "the number of country records."
    )

print(
    f"Records before transformation: "
    f"{bronze_count:,}"
)

print(
    f"Records after transformation:  "
    f"{classified_count:,}"
)

print(
    "Entity classification validation passed."
)

In [0]:
# Validate coordinate conversion

coordinate_validation_df = (
    bronze_countries_df
    .select(
        "country_id",
        "country_name",
        "longitude",
        "latitude"
    )

    .withColumn(
        "longitude_clean",
        F.when(
            F.trim(
                F.col("longitude")
            ) == "",
            None
        ).otherwise(
            F.col("longitude")
        )
    )

    .withColumn(
        "latitude_clean",
        F.when(
            F.trim(
                F.col("latitude")
            ) == "",
            None
        ).otherwise(
            F.col("latitude")
        )
    )

    .withColumn(
        "longitude_numeric",
        F.col(
            "longitude_clean"
        ).cast("double")
    )

    .withColumn(
        "latitude_numeric",
        F.col(
            "latitude_clean"
        ).cast("double")
    )
)

In [0]:
invalid_longitude_count = (
    coordinate_validation_df
    .filter(
        F.col("longitude_clean").isNotNull()
        & F.col("longitude_numeric").isNull()
    )
    .count()
)

invalid_latitude_count = (
    coordinate_validation_df
    .filter(
        F.col("latitude_clean").isNotNull()
        & F.col("latitude_numeric").isNull()
    )
    .count()
)

print(
    f"Invalid longitude values: "
    f"{invalid_longitude_count}"
)

print(
    f"Invalid latitude values: "
    f"{invalid_latitude_count}"
)

if invalid_longitude_count > 0:

    raise RuntimeError(
        "Non-empty longitude values failed "
        "numeric conversion."
    )

if invalid_latitude_count > 0:

    raise RuntimeError(
        "Non-empty latitude values failed "
        "numeric conversion."
    )

print(
    "Coordinate conversion validation passed."
)

In [0]:
# Validate geographic coordinate ranges

invalid_longitude_range = (
    silver_countries_df
    .filter(
        F.col("longitude").isNotNull()
        & (
            (F.col("longitude") < -180)
            | (F.col("longitude") > 180)
        )
    )
    .count()
)

invalid_latitude_range = (
    silver_countries_df
    .filter(
        F.col("latitude").isNotNull()
        & (
            (F.col("latitude") < -90)
            | (F.col("latitude") > 90)
        )
    )
    .count()
)

print(
    f"Out-of-range longitudes: "
    f"{invalid_longitude_range}"
)

print(
    f"Out-of-range latitudes: "
    f"{invalid_latitude_range}"
)

if invalid_longitude_range > 0:
    raise RuntimeError(
        "Invalid longitude range detected."
    )

if invalid_latitude_range > 0:
    raise RuntimeError(
        "Invalid latitude range detected."
    )

print(
    "Coordinate range validation passed."
)

In [0]:
# Validate Silver entity keys

null_entity_ids = (
    silver_countries_df
    .filter(
        F.col("entity_id").isNull()
        | (
            F.length(
                F.trim(
                    F.col("entity_id")
                )
            ) == 0
        )
    )
    .count()
)

null_entity_names = (
    silver_countries_df
    .filter(
        F.col("entity_name").isNull()
        | (
            F.length(
                F.trim(
                    F.col("entity_name")
                )
            ) == 0
        )
    )
    .count()
)

duplicate_entity_ids = (
    silver_countries_df
    .groupBy(
        "entity_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    f"Null/empty entity IDs: "
    f"{null_entity_ids}"
)

print(
    f"Null/empty entity names: "
    f"{null_entity_names}"
)

print(
    f"Duplicate entity IDs: "
    f"{duplicate_entity_ids}"
)

if (
    null_entity_ids > 0
    or null_entity_names > 0
    or duplicate_entity_ids > 0
):

    raise RuntimeError(
        "Silver entity key validation failed."
    )

print(
    "Silver entity key validation passed."
)

In [0]:
# Inspect individual economies

display(
    silver_countries_df
    .filter(
        F.col("entity_type")
        == "COUNTRY_OR_ECONOMY"
    )
    .select(
        "entity_id",
        "iso2_code",
        "entity_name",
        "region_name",
        "income_level_name",
        "capital_city"
    )
    .orderBy(
        "entity_name"
    )
    .limit(20)
)

In [0]:
# Inspect World Bank aggregates

display(
    silver_countries_df
    .filter(
        F.col("entity_type")
        == "AGGREGATE"
    )
    .select(
        "entity_id",
        "iso2_code",
        "entity_name",
        "region_name"
    )
    .orderBy(
        "entity_name"
    )
    .limit(30)
)

In [0]:
# Final pre-write validation

silver_count = (
    silver_countries_df.count()
)

aggregate_count = (
    silver_countries_df
    .filter(
        F.col("entity_type")
        == "AGGREGATE"
    )
    .count()
)

country_economy_count = (
    silver_countries_df
    .filter(
        F.col("entity_type")
        == "COUNTRY_OR_ECONOMY"
    )
    .count()
)

print("=" * 60)
print("SILVER COUNTRY VALIDATION")
print("=" * 60)

print(
    f"Bronze records:        "
    f"{bronze_count:,}"
)

print(
    f"Silver records:        "
    f"{silver_count:,}"
)

print(
    f"Countries/economies:   "
    f"{country_economy_count:,}"
)

print(
    f"Aggregates:            "
    f"{aggregate_count:,}"
)

print(
    f"Total classified:      "
    f"{country_economy_count + aggregate_count:,}"
)

if silver_count != bronze_count:

    raise RuntimeError(
        "Bronze/Silver row-count mismatch."
    )

if (
    country_economy_count
    + aggregate_count
    != silver_count
):

    raise RuntimeError(
        "Not all entities were classified."
    )

print(
    "\nAll Silver country validations passed."
)

In [0]:
# Write Silver countries table

(
    silver_countries_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Saved Silver countries to:"
    f"\n{TARGET_TABLE}"
)

In [0]:
# Read-back validation

saved_silver_df = spark.table(
    TARGET_TABLE
)

saved_count = (
    saved_silver_df.count()
)

saved_distinct_entities = (
    saved_silver_df
    .select(
        "entity_id"
    )
    .distinct()
    .count()
)

print(
    f"Expected records: "
    f"{silver_count:,}"
)

print(
    f"Saved records:    "
    f"{saved_count:,}"
)

print(
    f"Distinct entities:"
    f" {saved_distinct_entities:,}"
)

if saved_count != silver_count:

    raise RuntimeError(
        "Silver country write row-count "
        "validation failed."
    )

if saved_distinct_entities != saved_count:

    raise RuntimeError(
        "Silver country entity uniqueness "
        "validation failed."
    )

print(
    "Silver country write validated."
)

In [0]:
# Final summary

print("=" * 70)
print("WORLD BANK COUNTRY SILVER TRANSFORMATION")
print("=" * 70)

print(
    f"Source:              "
    f"{SOURCE_TABLE}"
)

print(
    f"Target:              "
    f"{TARGET_TABLE}"
)

print(
    f"Bronze records:      "
    f"{bronze_count:,}"
)

print(
    f"Silver records:      "
    f"{saved_count:,}"
)

print(
    f"Countries/economies: "
    f"{country_economy_count:,}"
)

print(
    f"Aggregates:          "
    f"{aggregate_count:,}"
)

print(
    f"Duplicate IDs:       "
    f"{duplicate_entity_ids:,}"
)

print(
    f"Invalid longitude:   "
    f"{invalid_longitude_count:,}"
)

print(
    f"Invalid latitude:    "
    f"{invalid_latitude_count:,}"
)

print(
    "Layer:               Silver"
)

print(
    "Status:              SUCCESS"
)